# NYC Mobility: Source Ingestion

## What ingestion means

Ingestion is how we bring source data into storage before transforming it. This notebook uses repeatable source downloads, deterministic filenames, content hashes, and metadata sidecars. Existing raw files are preserved; do not delete them just to create a clean-looking rerun.


## Shared idempotent landing helper

Run this cell first. It downloads a source file, compares its SHA-256 hash with any existing file at the deterministic path, and writes a metadata sidecar only when missing. Identical reruns do not change the landed file. A same-name file with different content is preserved and raised for review instead of being overwritten.


In [0]:
%python
import hashlib
import json
import requests
from datetime import datetime, timezone
from pathlib import Path

REQUEST_HEADERS = {
    "User-Agent": "FTW-B12-Data-Engineering-Course-Project"
}

def fetch_and_land(source_url, target_path, source_system, timeout=120):
    target_path = Path(target_path)
    target_path.parent.mkdir(parents=True, exist_ok=True)

    response = requests.get(
        source_url,
        timeout=timeout,
        headers=REQUEST_HEADERS,
    )
    response.raise_for_status()
    raw_content = response.content
    if not raw_content:
        raise ValueError(f"Empty response received from {source_url}")

    content_hash = hashlib.sha256(raw_content).hexdigest()
    if target_path.exists():
        existing_hash = hashlib.sha256(target_path.read_bytes()).hexdigest()
        if existing_hash != content_hash:
            raise RuntimeError(
                f"{target_path.name} already exists with different content; "
                "review before replacing preserved raw data."
            )
        action = "IDEMPOTENT_SKIP"
    else:
        target_path.write_bytes(raw_content)
        action = "WRITTEN"

    metadata_path = Path(f"{target_path}.metadata.json")
    if metadata_path.exists():
        existing_metadata = json.loads(metadata_path.read_text())
        if existing_metadata.get("sha256") != content_hash:
            raise RuntimeError(
                f"Metadata hash mismatch for {metadata_path.name}; review the source pair."
            )
    else:
        metadata = {
            "source_system": source_system,
            "source_url": source_url,
            "source_file": target_path.name,
            "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
            "http_status": response.status_code,
            "content_type": response.headers.get("Content-Type"),
            "bytes_received": len(raw_content),
            "sha256": content_hash,
        }
        metadata_path.write_text(json.dumps(metadata, indent=2))

    evidence = {
        "action": action,
        "http_status": response.status_code,
        "source_file": target_path.name,
        "output_path": str(target_path),
        "metadata_path": str(metadata_path),
        "bytes_received": len(raw_content),
        "sha256": content_hash,
    }
    print(json.dumps(evidence, indent=2))
    return evidence


## Green Taxi

NYC TLC publishes Green Taxi trip records as monthly Parquet files. Select March, April, or May with the widget, then land only that month using the official deterministic filename. This replaces the undocumented manual-only landing step.


In [0]:
%python
allowed_taxi_months = ["2026-03", "2026-04", "2026-05"]
try:
    green_taxi_month = dbutils.widgets.get("green_taxi_month").strip()
except Exception:
    dbutils.widgets.dropdown(
        "green_taxi_month",
        "2026-03",
        allowed_taxi_months,
        "Green Taxi month",
    )
    green_taxi_month = dbutils.widgets.get("green_taxi_month").strip()

if green_taxi_month not in allowed_taxi_months:
    raise ValueError(f"green_taxi_month must be one of {allowed_taxi_months}")

green_taxi_filename = f"green_tripdata_{green_taxi_month}.parquet"
green_taxi_url = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    f"{green_taxi_filename}"
)
green_taxi_path = (
    Path("/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/")
    / "groups/week-08/group-b/source/green_taxi"
    / green_taxi_filename
)

green_taxi_evidence = fetch_and_land(
    green_taxi_url,
    green_taxi_path,
    source_system="nyc_tlc",
)


The Green Taxi cell returned HTTP 200 for `green_tripdata_2026-03.parquet` and reported `IDEMPOTENT_SKIP`, confirming that the downloaded content matched the preserved file. The saved output includes the metadata path, 1,082,530-byte response size, and SHA-256 hash. April and May executions are not shown in this notebook.


## Taxi Zones

The Taxi Zone lookup is downloaded from the official NYC TLC link and saved with the exact filename `taxi_zone_lookup.csv`. The deterministic path and hash check make the landing step repeatable.


In [0]:
%python
taxi_zone_url = (
    "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
)
taxi_zone_path = (
    Path("/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/")
    / "groups/week-08/group-b/source/taxi_zones"
    / "taxi_zone_lookup.csv"
)

taxi_zone_evidence = fetch_and_land(
    taxi_zone_url,
    taxi_zone_path,
    source_system="nyc_tlc",
)


The Taxi Zone cell returned HTTP 200 for `taxi_zone_lookup.csv` and reported `IDEMPOTENT_SKIP`, confirming that the downloaded content matched the preserved file. The saved output includes the metadata path, 12,331-byte response size, and SHA-256 hash.


## Weather API ingestion

We use the Open-Meteo Archive API because the assignment period is historical: March through May 2026. Select one month with the `weather_month` widget so March, April, and May can be ingested independently. The response is saved as raw JSON without flattening because cleaning and reshaping belong after ingestion.


In [0]:
%python
import calendar
from datetime import datetime

allowed_months = ["2026-03", "2026-04", "2026-05"]
try:
    weather_month = dbutils.widgets.get("weather_month").strip()
except Exception:
    dbutils.widgets.dropdown("weather_month", "2026-03", allowed_months, "Weather month")
    weather_month = dbutils.widgets.get("weather_month").strip()

if weather_month not in allowed_months:
    raise ValueError(f"weather_month must be one of {allowed_months}")

month_start = datetime.strptime(weather_month, "%Y-%m").date()
month_end_day = calendar.monthrange(month_start.year, month_start.month)[1]
start_date = month_start.isoformat()
end_date = month_start.replace(day=month_end_day).isoformat()

weather_url = (
    f"https://archive-api.open-meteo.com/v1/archive"
    f"?latitude=40.7128"
    f"&longitude=-74.006"
    f"&start_date={start_date}"
    f"&end_date={end_date}"
    f"&hourly=temperature_2m,precipitation,rain,snowfall,weather_code,wind_speed_10m"
    f"&timezone=America%2FNew_York"
)

weather_filename = f"open_meteo_{start_date}_{end_date}.json"
weather_path = (
    Path("/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/")
    / "groups/week-08/group-b/source/weather"
    / weather_filename
)

print("Selected month:", weather_month)
print("Date range:", start_date, "to", end_date)
weather_evidence = fetch_and_land(
    weather_url,
    weather_path,
    source_system="open_meteo",
)


The Weather cell returned HTTP 200 for March 2026 and reported `WRITTEN`. It saved `open_meteo_2026-03-01_2026-03-31.json` plus its metadata sidecar, with 34,007 bytes and the SHA-256 hash shown in the cell output. April, May, and a repeated-month `IDEMPOTENT_SKIP` result are not shown in this notebook.


## Traffic Advisory web scraping

This is the optional bonus source. The scraper hashes the downloaded HTML before writing. If the same content already exists, it performs an idempotent skip. Only new content receives a new UTC timestamp batch ID, raw HTML file, and metadata JSON sidecar.

The verified saved batch is `20260914T040846Z`. We use these exact files for the Bronze load:

- `nyc_dot_weekend_traffic_20260914T040846Z.html`
- `nyc_dot_weekend_traffic_20260914T040846Z.metadata.json`

Scraper idempotency and Bronze-load idempotency are separate checks. The scraper skips an unchanged response by content hash. For the Bronze idempotency test, rerun the Bronze load against the same saved HTML and metadata files.


In [0]:
import hashlib
import json
import requests
from datetime import datetime, timezone
from pathlib import Path

source_url = "https://www.nyc.gov/html/dot/html/motorist/wkndtraf.shtml"

base_path = (
    "/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/"
    "groups/week-08/group-b/source/traffic_advisory"
)

advisory_dir = Path(base_path)
advisory_dir.mkdir(parents=True, exist_ok=True)

response = requests.get(
    source_url,
    timeout=60,
    headers={
        "User-Agent": "FTW-B12-Data-Engineering-Course-Project"
    }
)

response.raise_for_status()

raw_content = response.content
content_hash = hashlib.sha256(raw_content).hexdigest()
matching_metadata = None

for metadata_path in advisory_dir.glob("*.metadata.json"):
    try:
        existing_metadata = json.loads(metadata_path.read_text())
    except (OSError, json.JSONDecodeError) as exc:
        print(f"Warning: could not inspect {metadata_path.name}: {exc}")
        continue

    existing_hash = existing_metadata.get("sha256")
    if existing_hash is None:
        existing_html = metadata_path.with_name(
            metadata_path.name.replace(".metadata.json", ".html")
        )
        if existing_html.exists():
            existing_hash = hashlib.sha256(existing_html.read_bytes()).hexdigest()

    if existing_hash == content_hash:
        matching_metadata = metadata_path
        break

if matching_metadata is not None:
    action = "IDEMPOTENT_SKIP"
    metadata_path = matching_metadata
    existing_metadata = json.loads(metadata_path.read_text())
    batch_id = existing_metadata.get("batch_id")
    html_path = metadata_path.with_name(
        metadata_path.name.replace(".metadata.json", ".html")
    )
else:
    action = "WRITTEN"
    scraped_at = datetime.now(timezone.utc)
    batch_id = scraped_at.strftime("%Y%m%dT%H%M%SZ")
    html_path = advisory_dir / f"nyc_dot_weekend_traffic_{batch_id}.html"
    metadata_path = advisory_dir / f"nyc_dot_weekend_traffic_{batch_id}.metadata.json"

    html_path.write_bytes(raw_content)

    metadata = {
        "source_system": "nyc_dot",
        "source_url": source_url,
        "source_file": html_path.name,
        "batch_id": batch_id,
        "ingested_at_utc": scraped_at.isoformat(),
        "http_status": response.status_code,
        "content_type": response.headers.get("Content-Type"),
        "bytes_received": len(raw_content),
        "sha256": content_hash
    }
    metadata_path.write_text(json.dumps(metadata, indent=2))


traffic_evidence = {
    "action": action,
    "http_status": response.status_code,
    "batch_id": batch_id,
    "html_path": str(html_path),
    "metadata_path": str(metadata_path),
    "bytes_received": len(raw_content),
    "sha256": content_hash,
}
print(json.dumps(traffic_evidence, indent=2))

The Traffic Advisory cell returned HTTP 200 and reported `WRITTEN`. It created batch `20260914T124503Z`, saved the HTML and metadata sidecar, and recorded 53,391 bytes plus the SHA-256 hash shown in the cell output. A second-run `IDEMPOTENT_SKIP` result is not shown in this notebook. BeautifulSoup inspection and event parsing remain outside ingestion and Bronze.


## Ingestion Summary

Green Taxi and Taxi Zones now have repeatable downloads from the official NYC TLC file links. Green Taxi and Weather are parameterized by month. Deterministic filenames, metadata sidecars, and SHA-256 checks preserve raw files and make identical reruns visible as `IDEMPOTENT_SKIP`. Traffic Advisory timestamps only genuinely new content. Raw responses remain uncleaned and unexpanded in ingestion.

## Remaining Databricks evidence for the full assignment

The current notebook now contains execution evidence for Green Taxi March, Taxi Zones, Weather March, and one Traffic Advisory run. To demonstrate the full March-May and rerun requirements, still capture:

1. Green Taxi April and May outputs.
2. Weather April and May outputs.
3. A repeated Weather month showing `IDEMPOTENT_SKIP`.
4. A repeated Traffic Advisory run showing `IDEMPOTENT_SKIP` when the page content is unchanged.
5. The executed notebook outputs or screenshots as the group's agreed evidence.

Do not delete the existing raw source files. They are lineage evidence, and the code safely validates or skips them. Only the results visibly stored in the notebook are documented above.
